# 01 - Scraping Twitter/X: Sentimen Kebijakan Pendidikan Jakarta & Banten

Notebook ini mengambil data tweet publik terkait kebijakan pendidikan dan akses pendidikan tinggi
di DKI Jakarta dan Provinsi Banten menggunakan Apify Actor `kaitoeasyapi/twitter-x-data-tweet-scraper-pay-per-result-cheapest`.

## 1. Setup & Konfigurasi

In [ ]:
from apify_client import ApifyClient
from pathlib import Path
import pandas as pd
import time
import os

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

APIFY_API_TOKEN = os.getenv("APIFY_API_TOKEN")
if not APIFY_API_TOKEN or APIFY_API_TOKEN == "PASTE_TOKEN_KAMU_DI_SINI":
    raise ValueError("Isi APIFY_API_TOKEN di file .env dulu.")

client = ApifyClient(APIFY_API_TOKEN)

def get_run_field(run_obj, camel_key, snake_key):
    if isinstance(run_obj, dict):
        return run_obj[camel_key]
    if hasattr(run_obj, "model_dump"):
        data = run_obj.model_dump(by_alias=True)
        if camel_key in data:
            return data[camel_key]
    return getattr(run_obj, snake_key)

# Actor ID untuk kaitoeasyapi
ACTOR_ID = "kaitoeasyapi/twitter-x-data-tweet-scraper-pay-per-result-cheapest"

## 2. Definisi Keyword per Wilayah

Keyword disusun berdasarkan isu pendidikan spesifik di masing-masing wilayah,
sesuai kerangka esai (PPDB, KJP untuk Jakarta; akses sekolah, BOS untuk Banten).

In [ ]:
# Keyword Jakarta
keywords_jakarta = [
    "PPDB Jakarta",
    "KJP Jakarta",
    "KJMU Jakarta",
    "zonasi sekolah Jakarta",
    "sekolah negeri Jakarta",
    "biaya kuliah Jakarta",
    "UKT Jakarta",
    "beasiswa kuliah Jakarta",
]

# Keyword Banten
keywords_banten = [
    "putus sekolah Banten",
    "akses sekolah Banten",
    "akses sekolah Serang",
    "akses sekolah Pandeglang",
    "akses sekolah Lebak",
    "BOS Banten",
    "beasiswa Banten",
    "beasiswa kuliah Banten",
    "biaya kuliah Banten",
    "PPDB Banten",
]

# Gabungkan dengan label wilayah untuk tracking
all_queries = (
    [(kw, "Jakarta") for kw in keywords_jakarta] +
    [(kw, "Banten") for kw in keywords_banten]
)

print(f"Total {len(all_queries)} query akan dijalankan:")
for kw, region in all_queries:
    print(f"  - [{region}] {kw}")

## 3. Test Run

Test dengan 1 query dan limit kecil dulu untuk memastikan format input Actor benar dan menghindari biaya tidak perlu kalau ada kesalahan konfigurasi.

In [ ]:
# Test dengan 1 keyword, limit 10 tweet saja
test_input = {
    "searchTerms": ["PPDB Jakarta"],
    "maxItems": 10,
    "queryType": "Latest",  # atau "Top" jika hasil terlalu sedikit
    "lang": "in",
}

print("Menjalankan test run...")
run = client.actor(ACTOR_ID).call(run_input=test_input)
test_dataset_id = get_run_field(run, "defaultDatasetId", "default_dataset_id")

print(f"Run selesai. Status: {get_run_field(run, 'status', 'status')}")
print(f"Dataset ID: {test_dataset_id}")

In [ ]:
# Lihat hasil test
test_items = list(client.dataset(test_dataset_id).iterate_items())
print(f"Jumlah tweet didapat: {len(test_items)}")

if len(test_items) > 0:
    print("\nContoh field yang tersedia:")
    print(list(test_items[0].keys()))
    print("\nContoh isi tweet pertama:")
    print(test_items[0])
else:
    print("⚠️ Tidak ada hasil. Cek kembali format input atau coba queryType='Top'")

## 4. Full Scraping — Semua Keyword

Setelah test berhasil dan kamu yakin field-nya sesuai, lanjut ke scraping penuh.

**Estimasi biaya:** dengan 18 keyword pada tarif $0.18/1000 tweet:
- `maxItems=300` -> maksimal ~5.400 tweet -> sekitar **$0.97**
- `maxItems=500` -> maksimal ~9.000 tweet -> sekitar **$1.62**
- `maxItems=1000` -> maksimal ~18.000 tweet -> sekitar **$3.24**
- `maxItems=1200` -> maksimal ~21.600 tweet -> sekitar **$3.89**

Sesuaikan `MAX_ITEMS_PER_QUERY` sesuai budget kamu.

In [ ]:
MAX_ITEMS_PER_QUERY = 1200  # 18 keyword x 1200 = maksimal 21.600 tweet (~$3.89)

all_results = []

for keyword, region in all_queries:
    print(f"\nScraping: [{region}] '{keyword}' ...")
    
    run_input = {
        "searchTerms": [keyword],
        "maxItems": MAX_ITEMS_PER_QUERY,
        "queryType": "Latest",
        "lang": "in",
    }
    
    try:
        run = client.actor(ACTOR_ID).call(run_input=run_input)
        dataset_id = get_run_field(run, "defaultDatasetId", "default_dataset_id")
        items = list(client.dataset(dataset_id).iterate_items())
        
        # Tambahkan metadata keyword & region ke setiap tweet
        for item in items:
            item["_search_keyword"] = keyword
            item["_region"] = region
        
        all_results.extend(items)
        print(f"  -> Didapat {len(items)} tweet")
        
    except Exception as e:
        print(f"  -> ERROR: {e}")
        continue
    
    # Delay sopan antar request
    time.sleep(3)

print(f"\n=== TOTAL: {len(all_results)} tweet terkumpul ===")

## 5. Simpan ke DataFrame & CSV

In [ ]:
df = pd.DataFrame(all_results)

if df.empty:
    raise ValueError("Tidak ada tweet terkumpul. Cek keyword, token, atau ubah queryType ke 'Top'.")

# Pecah metadata author agar mudah dicek tanpa error karena kolom author berisi dict.
if "author" in df.columns:
    df["author_username"] = df["author"].apply(lambda x: x.get("userName") if isinstance(x, dict) else None)
    df["author_name"] = df["author"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)
    df["author_followers"] = df["author"].apply(lambda x: x.get("followers") if isinstance(x, dict) else None)

print(f"Shape: {df.shape}")
print(f"\nDistribusi per wilayah:")
print(df["_region"].value_counts())
print(f"\nDistribusi per keyword:")
print(df["_search_keyword"].value_counts())

df.head()

In [ ]:
# Cek duplikasi sebelum simpan (penting untuk tahap filter bot nanti)
id_col = "id" if "id" in df.columns else None
n_duplicate = df.duplicated(subset=[id_col]).sum() if id_col else 0
print(f"Jumlah duplikat terdeteksi: {n_duplicate}")

df_clean = df.drop_duplicates(subset=[id_col]) if id_col else df.copy()
print(f"Total setelah dedup: {len(df_clean)}")

# Simpan raw data (sebelum filter bot/spam - itu tahap berikutnya)
output_path = "raw_tweets_jakarta_banten.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nData disimpan ke: {output_path}")

## 6. Cek Cepat Distribusi Data

Sebelum lanjut ke notebook berikutnya (filter bot & spam), cek dulu kualitas data mentah:
apakah ada akun yang muncul terlalu sering (indikasi bot), atau isi tweet yang sangat mirip.

In [ ]:
# Cari kolom username (nama field bisa beda-beda tergantung Actor)
preferred_username_cols = ["author_username", "username", "userName", "screen_name"]
username_candidates = [c for c in preferred_username_cols if c in df_clean.columns]
print(f"Kolom kandidat username: {username_candidates}")

if username_candidates:
    col = username_candidates[0]
    print(f"\nTop 10 akun paling sering muncul (kolom: {col}):")
    print(df_clean[col].value_counts().head(10))
    print("\n⚠️ Akun dengan frekuensi sangat tinggi perlu diperiksa di tahap filter bot selanjutnya.")

## Langkah Selanjutnya

1. Lanjut ke `02_bot_filter.ipynb` untuk Data Verification Layer (filter spam, bot, CIB)
2. Cek manual beberapa sample tweet untuk memastikan relevansi dengan topik pendidikan
3. Jika hasil terlalu sedikit per keyword, coba ubah `queryType` ke `"Top"` atau perluas keyword